In [3]:
#  DATA PROFILING


# Import libraries for data analysis
import pandas as pd
import numpy as np

# Import libraries for visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Import library for managing file paths
from pathlib import Path

# Configure display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

# Configure chart appearance
sns.set_theme(style="whitegrid")

# Identify the project directory
cwd = Path.cwd()

if (cwd / "data").is_dir():
    PROJECT_ROOT = cwd

elif (cwd.parent / "data").is_dir():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Open the notebook from the project root "
        "or notebooks folder."
    )

# Create folders for profiling reports
REPORT_DIR = PROJECT_ROOT / "reports" / "profiling"
FIGURE_DIR = REPORT_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Confirm successful configuration
print("Project root:", PROJECT_ROOT)
print("Report folder:", REPORT_DIR)
print("Figure folder:", FIGURE_DIR)
print("Libraries imported successfully.")

Project root: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction
Report folder: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\profiling
Figure folder: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\profiling\figures
Libraries imported successfully.


In [4]:

# LOAD THE ORIGINAL DATASET

# Define the location of the original dataset
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Outpatient_appointments_synthetic.csv"
)

# Check that the dataset exists
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

# Load the original dataset
df = pd.read_csv(DATA_PATH)

# Preserve an untouched copy of the original data
df_raw = df.copy(deep=True)

# Confirm successful loading
print("Dataset loaded successfully.")

# Display the first five records
display(df.head())

Dataset loaded successfully.


,appointment_id,patient_id,age,sex,imd_decile,ethnic_group,distance_km,interpreter_required,long_term_conditions,specialty,appointment_type,consultation_mode,referral_source,appointment_datetime,booked_datetime,rescheduled_count,sms_reminder_sent,previous_dnas_12m,attendance_status,clinic_checkin_time
0,OPA2004425,P113753,70,M,2.0,Not Stated,5.7,N,1,Respiratory Medicine,Follow-up,Video,Consultant,02/09/2025 11:30,07/06/2025 12:59,1,Y,1,Attended,11:16
1,OPA2000901,P119960,58,Female,2.0,White,4.7,N,0,Cardiology,Follow-up,Face-to-face,Consultant,12/11/2025 10:30,13/08/2025 12:33,0,Y,0,Attended,10:13
2,OPA2014471,P122474,76,Female,9.0,White,18.1,N,1,Diabetes and Endocrinology,New,Face-to-face,Other,17/11/2025 13:30,03/10/2025 14:08,0,Y,0,Cancelled by hospital,NaN
3,OPA2002727,P113052,69,Male,5.0,White,13.1,N,3,Trauma and Orthopaedics,Follow-up,Face-to-face,Consultant,14/07/2025 14:30,05/06/2025 16:40,1,Y,0,Attended,14:22
4,OPA2007989,P120866,72,Female,9.0,White,4.1,N,1,Opthalmology,Follow-up,Face-to-face,Consultant,13/10/2025 10:00,27/08/2025 17:14,0,Y,0,Attended,09:38


In [5]:

# EXAMINE DATASET DIMENSIONS AND COLUMN NAMES


# Purpose:
# Examine the size and structure of the original dataset.
# Identify the number of records, columns and duplicate
# column names before proceeding with data quality checks.

# 1. Identify the number of rows and columns
rows, columns = df.shape

print("Number of rows:", rows)
print("Number of columns:", columns)

# 2. Calculate the total number of data cells
total_cells = rows * columns

print("Total data cells:", total_cells)

# 3. Display all column names
print("\nDATASET COLUMN NAMES")

for number, column in enumerate(df.columns, start=1):
    print(f"{number}. {column}")

# 4. Check for duplicate column names
duplicate_columns = df.columns.duplicated().sum()

print("\nNumber of duplicate column names:", duplicate_columns)

Number of rows: 40180
Number of columns: 20
Total data cells: 803600

DATASET COLUMN NAMES
1. appointment_id
2. patient_id
3. age
4. sex
5. imd_decile
6. ethnic_group
7. distance_km
8. interpreter_required
9. long_term_conditions
10. specialty
11. appointment_type
12. consultation_mode
13. referral_source
14. appointment_datetime
15. booked_datetime
16. rescheduled_count
17. sms_reminder_sent
18. previous_dnas_12m
19. attendance_status
20. clinic_checkin_time

Number of duplicate column names: 0


In [6]:

# EXAMINE DATA TYPES AND DATASET STRUCTURE


# Purpose:
# Examine the data types, completeness and unique values
# of each column in the original dataset.
#
# This helps identify potential data quality issues,
# including incorrectly stored dates and categorical fields.
#


# 1. Display general dataset information
print("DATASET STRUCTURE")
print("=" * 60)

df.info()

# 2. Generate a detailed column summary
dtype_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null Count": df.notna().sum().values,
    "Missing Count": df.isna().sum().values,
    "Missing Percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "Unique Values": df.nunique(dropna=True).values
})

# 3. Display the summary
print("\nCOLUMN STRUCTURE AND DATA QUALITY SUMMARY")
display(dtype_summary)

# 4. Identify columns stored as object types
object_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nCOLUMNS STORED AS OBJECT:")
for column in object_columns:
    print("-", column)

# 5. Save the summary for documentation
dtype_summary.to_csv(
    REPORT_DIR / "column_structure.csv",
    index=False
)

print("\nColumn structure report saved successfully.")

DATASET STRUCTURE
<class 'pandas.DataFrame'>
RangeIndex: 40180 entries, 0 to 40179
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        40180 non-null  str    
 1   patient_id            40180 non-null  str    
 2   age                   40180 non-null  int64  
 3   sex                   40180 non-null  str    
 4   imd_decile            38770 non-null  float64
 5   ethnic_group          40180 non-null  str    
 6   distance_km           39777 non-null  float64
 7   interpreter_required  40180 non-null  str    
 8   long_term_conditions  40180 non-null  int64  
 9   specialty             40180 non-null  str    
 10  appointment_type      40180 non-null  str    
 11  consultation_mode     40180 non-null  str    
 12  referral_source       40180 non-null  str    
 13  appointment_datetime  40180 non-null  str    
 14  booked_datetime       40180 non-null  str    
 15  rescheduled_

,Column,Data Type,Non-Null Count,Missing Count,Missing Percentage,Unique Values
0,appointment_id,str,40180,0,0.00,40000
1,patient_id,str,40180,0,0.00,15458
2,age,int64,40180,0,0.00,106
3,sex,str,40180,0,0.00,7
4,imd_decile,float64,38770,1410,3.51,10
5,ethnic_group,str,40180,0,0.00,6
6,distance_km,float64,39777,403,1.00,349
7,interpreter_required,str,40180,0,0.00,2
8,long_term_conditions,int64,40180,0,0.00,9
9,specialty,str,40180,0,0.00,9



COLUMNS STORED AS OBJECT:
- appointment_id
- patient_id
- sex
- ethnic_group
- interpreter_required
- specialty
- appointment_type
- consultation_mode
- referral_source
- appointment_datetime
- booked_datetime
- sms_reminder_sent
- attendance_status
- clinic_checkin_time

Column structure report saved successfully.


C:\Users\User.DESKTOP-775\AppData\Local\Temp\ipykernel_48612\1014491578.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(


In [7]:

# EXAMINE DATA TYPES AND DATASET STRUCTURE


# Purpose:
# Examine the original dataset to identify:
# 1. Data types of all columns
# 2. Number of non-missing values
# 3. Number of missing values
# 4. Percentage of missing values
# 5. Number of unique values
# 6. Columns stored as text
#




# 1. DISPLAY GENERAL DATASET INFORMATION

print("DATASET STRUCTURE")
print("=" * 60)

df.info()


# 2. GENERATE A DETAILED COLUMN SUMMARY

dtype_summary = pd.DataFrame({
    "Column": df.columns,

    "Data Type": df.dtypes.astype(str).values,

    "Non-Null Count": df.notna().sum().values,

    "Missing Count": df.isna().sum().values,

    "Missing Percentage": (
        df.isna().mean().values * 100
    ).round(2),

    "Unique Values": df.nunique(dropna=True).values
})


# 3. DISPLAY COLUMN STRUCTURE AND QUALITY SUMMARY

print("\nCOLUMN STRUCTURE AND DATA QUALITY SUMMARY")
print("=" * 60)

display(dtype_summary)


# 4. IDENTIFY TEXT AND CATEGORICAL COLUMNS

# Include object and string data types.


text_columns = df.select_dtypes(
    include=["object", "str", "string"]
).columns.tolist()

print("\nTEXT AND CATEGORICAL COLUMNS")
print("=" * 60)

for column in text_columns:
    print("-", column)


# 5. IDENTIFY COLUMNS WITH MISSING VALUES

missing_columns = dtype_summary[
    dtype_summary["Missing Count"] > 0
].copy()

print("\nCOLUMNS CONTAINING MISSING VALUES")
print("=" * 60)

if missing_columns.empty:
    print("No missing values identified.")
else:
    display(
        missing_columns[
            [
                "Column",
                "Missing Count",
                "Missing Percentage"
            ]
        ].sort_values(
            by="Missing Count",
            ascending=False
        )
    )


# 6. SAVE COLUMN STRUCTURE REPORT

dtype_summary.to_csv(
    REPORT_DIR / "column_structure.csv",
    index=False
)

print("\nColumn structure report saved successfully.")


# 7. CONFIRM PROFILING COMPLETION

print("\nCELL 4 COMPLETED SUCCESSFULLY")
print("Original dataset remains unchanged.")

DATASET STRUCTURE
<class 'pandas.DataFrame'>
RangeIndex: 40180 entries, 0 to 40179
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        40180 non-null  str    
 1   patient_id            40180 non-null  str    
 2   age                   40180 non-null  int64  
 3   sex                   40180 non-null  str    
 4   imd_decile            38770 non-null  float64
 5   ethnic_group          40180 non-null  str    
 6   distance_km           39777 non-null  float64
 7   interpreter_required  40180 non-null  str    
 8   long_term_conditions  40180 non-null  int64  
 9   specialty             40180 non-null  str    
 10  appointment_type      40180 non-null  str    
 11  consultation_mode     40180 non-null  str    
 12  referral_source       40180 non-null  str    
 13  appointment_datetime  40180 non-null  str    
 14  booked_datetime       40180 non-null  str    
 15  rescheduled_

,Column,Data Type,Non-Null Count,Missing Count,Missing Percentage,Unique Values
0,appointment_id,str,40180,0,0.00,40000
1,patient_id,str,40180,0,0.00,15458
2,age,int64,40180,0,0.00,106
3,sex,str,40180,0,0.00,7
4,imd_decile,float64,38770,1410,3.51,10
5,ethnic_group,str,40180,0,0.00,6
6,distance_km,float64,39777,403,1.00,349
7,interpreter_required,str,40180,0,0.00,2
8,long_term_conditions,int64,40180,0,0.00,9
9,specialty,str,40180,0,0.00,9



TEXT AND CATEGORICAL COLUMNS
- appointment_id
- patient_id
- sex
- ethnic_group
- interpreter_required
- specialty
- appointment_type
- consultation_mode
- referral_source
- appointment_datetime
- booked_datetime
- sms_reminder_sent
- attendance_status
- clinic_checkin_time

COLUMNS CONTAINING MISSING VALUES


,Column,Missing Count,Missing Percentage
19,clinic_checkin_time,6986,17.39
4,imd_decile,1410,3.51
6,distance_km,403,1.00



Column structure report saved successfully.

CELL 4 COMPLETED SUCCESSFULLY
Original dataset remains unchanged.


In [8]:

# STATISTICAL SUMMARY OF THE ORIGINAL DATASET


# Purpose:
# Examine numerical and categorical distributions,
# including negative values, without changing the dataset.


# 1. NUMERICAL STATISTICAL SUMMARY

print("NUMERICAL STATISTICAL SUMMARY")
print("=" * 60)

numerical_columns = df.select_dtypes(
    include=["number"]
).columns

numerical_summary = df[numerical_columns].describe().T

display(numerical_summary)


# 2. CATEGORICAL STATISTICAL SUMMARY

print("\nCATEGORICAL STATISTICAL SUMMARY")
print("=" * 60)

categorical_columns = df.select_dtypes(
    include=["object", "str", "string", "category"]
).columns

categorical_summary = df[categorical_columns].describe().T

display(categorical_summary)


# 3. NEGATIVE VALUE CHECK

print("\nNEGATIVE VALUE CHECK")
print("=" * 60)

negative_summary = (
    df[numerical_columns] < 0
).sum().to_frame(
    name="Negative Value Count"
)

display(negative_summary)


# 4. SAVE STATISTICAL SUMMARIES

numerical_summary.to_csv(
    REPORT_DIR / "numerical_summary.csv"
)

categorical_summary.to_csv(
    REPORT_DIR / "categorical_summary.csv"
)

negative_summary.to_csv(
    REPORT_DIR / "negative_value_summary.csv"
)


# 5. CONFIRM COMPLETION

print("\nCELL 5 COMPLETED SUCCESSFULLY")
print("Original dataset remains unchanged.")

NUMERICAL STATISTICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
age,40180.0,52.509657,29.106939,-1.0,31.0,58.0,74.0,999.0
imd_decile,38770.0,4.837787,2.812179,1.0,2.0,5.0,7.0,10.0
distance_km,39777.0,8.341851,5.684067,0.3,4.2,7.0,11.1,47.4
long_term_conditions,40180.0,1.281210,1.401055,0.0,0.0,1.0,2.0,8.0
rescheduled_count,40180.0,0.362842,0.683511,0.0,0.0,0.0,1.0,3.0
previous_dnas_12m,40180.0,0.266476,0.560660,0.0,0.0,0.0,0.0,8.0



CATEGORICAL STATISTICAL SUMMARY


,count,unique,top,freq
appointment_id,40180,40000,OPA2018043,2
patient_id,40180,15458,P100914,21
sex,40180,7,Female,20514
ethnic_group,40180,6,White,29275
interpreter_required,40180,2,N,38494
specialty,40180,9,Trauma and Orthopaedics,8849
appointment_type,40180,2,Follow-up,25011
consultation_mode,40180,3,Face-to-face,32220
referral_source,40180,4,Consultant,27179
appointment_datetime,40180,4427,08/12/2025 12:30,37



NEGATIVE VALUE CHECK


,Negative Value Count
age,35
imd_decile,0
distance_km,0
long_term_conditions,0
rescheduled_count,0
previous_dnas_12m,0



CELL 5 COMPLETED SUCCESSFULLY
Original dataset remains unchanged.


In [9]:

# CELL 6: DETAILED DATA QUALITY CHECKS


# Purpose:
# 1. Identify exact duplicate records
# 2. Investigate repeated appointment IDs
# 3. Examine invalid and unusual ages
# 4. Examine inconsistent sex categories
# 5. Check numerical ranges
#
# IMPORTANT:
# This cell performs data profiling only.
# No records are removed or modified.



# 1. CHECK EXACT DUPLICATE RECORDS

print("EXACT DUPLICATE RECORD CHECK")
print("=" * 60)

exact_duplicates = df.duplicated().sum()

print("Total records:", len(df))
print("Exact duplicate records:", exact_duplicates)


# 2. CHECK REPEATED APPOINTMENT IDs

print("\nREPEATED APPOINTMENT ID CHECK")
print("=" * 60)

duplicate_id_mask = df.duplicated(
    subset=["appointment_id"],
    keep=False
)

duplicate_id_records = df.loc[
    duplicate_id_mask
].copy()

duplicate_id_count = (
    df["appointment_id"].value_counts().gt(1).sum()
)

print("Unique appointment IDs:", df["appointment_id"].nunique())
print("Appointment IDs appearing more than once:", duplicate_id_count)
print("Records with repeated appointment IDs:", len(duplicate_id_records))

print("\nSAMPLE RECORDS WITH REPEATED APPOINTMENT IDs")

display(
    duplicate_id_records.sort_values(
        by="appointment_id"
    ).head(20)
)


# 3. INVESTIGATE INVALID AGES

print("\nAGE QUALITY CHECK")
print("=" * 60)

print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())

negative_ages = df[df["age"] < 0]

print("Records with negative ages:", len(negative_ages))

print("\nNEGATIVE AGE VALUES")

display(
    negative_ages["age"]
    .value_counts()
    .sort_index()
    .to_frame(name="Count")
)

print("\nSAMPLE RECORDS WITH NEGATIVE AGES")

display(
    negative_ages[
        ["appointment_id", "patient_id", "age", "attendance_status"]
    ].head(20)
)


# 4. INVESTIGATE UNUSUALLY HIGH AGES

print("\nUNUSUALLY HIGH AGE CHECK")
print("=" * 60)

# This is an investigation threshold, not a deletion rule.
high_age_records = df[df["age"] > 110]

print("Records with age greater than 110:", len(high_age_records))

display(
    high_age_records[
        ["appointment_id", "patient_id", "age"]
    ].head(20)
)


# 5. EXAMINE SEX CATEGORIES

print("\nSEX CATEGORY CHECK")
print("=" * 60)

sex_summary = (
    df["sex"]
    .value_counts(dropna=False)
    .rename_axis("Sex")
    .reset_index(name="Count")
)

sex_summary["Percentage"] = (
    sex_summary["Count"] / len(df) * 100
).round(2)

display(sex_summary)


# 6. CHECK NUMERICAL RANGES

print("\nNUMERICAL RANGE CHECK")
print("=" * 60)

numerical_range_summary = pd.DataFrame({
    "Minimum": df[numerical_columns].min(),
    "Maximum": df[numerical_columns].max(),
    "Missing Values": df[numerical_columns].isna().sum()
})

display(numerical_range_summary)


# 7. SAVE DATA QUALITY REPORTS

duplicate_id_records.to_csv(
    REPORT_DIR / "repeated_appointment_ids.csv",
    index=False
)

negative_ages.to_csv(
    REPORT_DIR / "negative_age_records.csv",
    index=False
)

sex_summary.to_csv(
    REPORT_DIR / "sex_category_summary.csv",
    index=False
)

numerical_range_summary.to_csv(
    REPORT_DIR / "numerical_range_summary.csv"
)


# 8. CONFIRM COMPLETION

print("\nDATA QUALITY REPORTS SAVED SUCCESSFULLY")
print("CELL 6 COMPLETED SUCCESSFULLY")
print("Original dataset remains unchanged.")

EXACT DUPLICATE RECORD CHECK
Total records: 40180
Exact duplicate records: 180

REPEATED APPOINTMENT ID CHECK
Unique appointment IDs: 40000
Appointment IDs appearing more than once: 180
Records with repeated appointment IDs: 360

SAMPLE RECORDS WITH REPEATED APPOINTMENT IDs


,appointment_id,patient_id,age,sex,imd_decile,ethnic_group,distance_km,interpreter_required,long_term_conditions,specialty,appointment_type,consultation_mode,referral_source,appointment_datetime,booked_datetime,rescheduled_count,sms_reminder_sent,previous_dnas_12m,attendance_status,clinic_checkin_time
33534,OPA2000027,P102269,78,M,1.0,White,11.5,N,0,Dermatology,Follow-up,Face-to-face,Consultant,17/09/2025 08:30,03/09/2025 09:49,0,Y,0,DNA,NaN
31355,OPA2000027,P102269,78,M,1.0,White,11.5,N,0,Dermatology,Follow-up,Face-to-face,Consultant,17/09/2025 08:30,03/09/2025 09:49,0,Y,0,DNA,NaN
2508,OPA2000586,P101579,73,Male,1.0,Not Stated,2.9,N,6,Diabetes and Endocrinology,New,Face-to-face,GP,11/08/2025 15:00,29/05/2025 13:55,0,Y,0,Cancelled by patient,NaN
38327,OPA2000586,P101579,73,Male,1.0,Not Stated,2.9,N,6,Diabetes and Endocrinology,New,Face-to-face,GP,11/08/2025 15:00,29/05/2025 13:55,0,Y,0,Cancelled by patient,NaN
5036,OPA2000616,P118126,53,Male,1.0,White,12.0,N,0,Trauma and Orthopaedics,New,Face-to-face,GP,04/11/2025 16:30,31/08/2025 10:59,1,N,1,Attended,16:26
30356,OPA2000616,P118126,53,Male,1.0,White,12.0,N,0,Trauma and Orthopaedics,New,Face-to-face,GP,04/11/2025 16:30,31/08/2025 10:59,1,N,1,Attended,16:26
12808,OPA2000733,P105300,88,Male,5.0,White,8.2,N,4,Cardiology,Follow-up,Face-to-face,Consultant,12/12/2025 16:30,30/11/2025 10:38,0,Y,0,Attended,16:21
13434,OPA2000733,P105300,88,Male,5.0,White,8.2,N,4,Cardiology,Follow-up,Face-to-face,Consultant,12/12/2025 16:30,30/11/2025 10:38,0,Y,0,Attended,16:21
10949,OPA2000778,P114587,78,Female,1.0,Asian or Asian British,3.2,N,2,Opthalmology,Follow-up,Video,Consultant,14/01/2026 09:00,23/12/2025 13:45,1,Y,0,Attended,09:11
24243,OPA2000778,P114587,78,Female,1.0,Asian or Asian British,3.2,N,2,Opthalmology,Follow-up,Video,Consultant,14/01/2026 09:00,23/12/2025 13:45,1,Y,0,Attended,09:11



AGE QUALITY CHECK
Minimum age: -1
Maximum age: 999
Records with negative ages: 35

NEGATIVE AGE VALUES


,Count
age,
-1,35



SAMPLE RECORDS WITH NEGATIVE AGES


,appointment_id,patient_id,age,attendance_status
112,OPA2029740,P108262,-1,Attended
450,OPA2025852,P101313,-1,Attended
710,OPA2011607,P105638,-1,Attended
1068,OPA2022893,P102698,-1,Attended
1394,OPA2021020,P103686,-1,Attended
1705,OPA2005193,P102854,-1,Cancelled by patient
2023,OPA2028687,P107032,-1,Attended
2820,OPA2006812,P113739,-1,Attended
8126,OPA2014960,P102857,-1,Cancelled by patient
8550,OPA2005871,P122563,-1,Attended


... (3 outputs left out)

,Sex,Count,Percentage
0,Female,20514,51.06
1,Male,18260,45.45
2,F,366,0.91
3,M,338,0.84
4,female,307,0.76
5,male,293,0.73
6,Unknown,102,0.25



NUMERICAL RANGE CHECK


,Minimum,Maximum,Missing Values
age,-1.0,999.0,0
imd_decile,1.0,10.0,1410
distance_km,0.3,47.4,403
long_term_conditions,0.0,8.0,0
rescheduled_count,0.0,3.0,0
previous_dnas_12m,0.0,8.0,0



DATA QUALITY REPORTS SAVED SUCCESSFULLY
CELL 6 COMPLETED SUCCESSFULLY
Original dataset remains unchanged.


In [10]:

# ATTENDANCE STATUS AND TARGET VARIABLE INVESTIGATION


# Purpose:
# 1. Examine all attendance outcome categories
# 2. Calculate counts and percentages
# 3. Investigate missing clinic check-in times by outcome
# 4. Identify potential target leakage




# 1. ATTENDANCE STATUS DISTRIBUTION

print("ATTENDANCE STATUS DISTRIBUTION")
print("=" * 60)

attendance_summary = (
    df["attendance_status"]
    .value_counts(dropna=False)
    .rename_axis("Attendance Status")
    .reset_index(name="Count")
)

attendance_summary["Percentage"] = (
    attendance_summary["Count"] / len(df) * 100
).round(2)

display(attendance_summary)


# 2. CHECK MISSING ATTENDANCE STATUS

print("\nMISSING ATTENDANCE STATUS CHECK")
print("=" * 60)

missing_attendance = df["attendance_status"].isna().sum()

print("Missing attendance status:", missing_attendance)


# 3. CHECK CLINIC CHECK-IN TIME BY ATTENDANCE STATUS

print("\nCHECK-IN TIME BY ATTENDANCE STATUS")
print("=" * 60)

checkin_summary = (
    df.groupby("attendance_status", dropna=False)
    ["clinic_checkin_time"]
    .agg(
        Total_Records="size",
        Checkin_Present="count"
    )
    .reset_index()
)

checkin_summary["Checkin_Missing"] = (
    checkin_summary["Total_Records"]
    - checkin_summary["Checkin_Present"]
)

checkin_summary["Missing_Percentage"] = (
    checkin_summary["Checkin_Missing"]
    / checkin_summary["Total_Records"]
    * 100
).round(2)

display(checkin_summary)


# 4. CHECK ATTENDED APPOINTMENTS WITHOUT CHECK-IN TIMES

print("\nATTENDED APPOINTMENTS WITHOUT CHECK-IN TIMES")
print("=" * 60)

attended_missing_checkin = df[
    (df["attendance_status"] == "Attended")
    & (df["clinic_checkin_time"].isna())
]

print(
    "Attended appointments missing check-in time:",
    len(attended_missing_checkin)
)


# 5. CHECK NON-ATTENDED APPOINTMENTS WITH CHECK-IN TIMES

print("\nNON-ATTENDED APPOINTMENTS WITH CHECK-IN TIMES")
print("=" * 60)

non_attended_with_checkin = df[
    (df["attendance_status"] != "Attended")
    & (df["clinic_checkin_time"].notna())
]

print(
    "Non-attended appointments with check-in time:",
    len(non_attended_with_checkin)
)


# 6. CHECK FOR INCONSISTENT ATTENDANCE LABELS

print("\nUNIQUE ATTENDANCE STATUS VALUES")
print("=" * 60)

for status in df["attendance_status"].unique():
    print(repr(status))


# 7. SAVE PROFILING RESULTS

attendance_summary.to_csv(
    REPORT_DIR / "attendance_status_summary.csv",
    index=False
)

checkin_summary.to_csv(
    REPORT_DIR / "checkin_by_attendance_status.csv",
    index=False
)


# 8. CONFIRM COMPLETION

print("\nCELL 7 COMPLETED SUCCESSFULLY")
print("Attendance profiling reports saved.")
print("Original dataset remains unchanged.")

ATTENDANCE STATUS DISTRIBUTION


,Attendance Status,Count,Percentage
0,Attended,33194,82.61
1,DNA,3329,8.29
2,Cancelled by patient,2433,6.06
3,Cancelled by hospital,1224,3.05



MISSING ATTENDANCE STATUS CHECK
Missing attendance status: 0

CHECK-IN TIME BY ATTENDANCE STATUS


,attendance_status,Total_Records,Checkin_Present,Checkin_Missing,Missing_Percentage
0,Attended,33194,33194,0,0.0
1,Cancelled by hospital,1224,0,1224,100.0
2,Cancelled by patient,2433,0,2433,100.0
3,DNA,3329,0,3329,100.0



ATTENDED APPOINTMENTS WITHOUT CHECK-IN TIMES
Attended appointments missing check-in time: 0

NON-ATTENDED APPOINTMENTS WITH CHECK-IN TIMES
Non-attended appointments with check-in time: 0

UNIQUE ATTENDANCE STATUS VALUES
'Attended'
'Cancelled by hospital'
'Cancelled by patient'
'DNA'

CELL 7 COMPLETED SUCCESSFULLY
Attendance profiling reports saved.
Original dataset remains unchanged.


In [11]:

# APPOINTMENT DATE AND BOOKING LEAD TIME PROFILING


# Purpose:
# 1. Validate appointment and booking dates
# 2. Identify missing or invalid dates
# 3. Check for appointments booked after their scheduled time
# 4. Examine booking lead-time distributions
# 5. Identify unusual booking intervals
# 6. Save profiling results



import pandas as pd

print("APPOINTMENT DATE AND BOOKING LEAD TIME PROFILING")
print("=" * 60)


# 1. CONVERT DATE COLUMNS FOR INVESTIGATION ONLY


appointment_dates = pd.to_datetime(
    df["appointment_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

booking_dates = pd.to_datetime(
    df["booked_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)


# 2. CHECK INVALID OR MISSING DATES

print("\nDATE VALIDATION CHECK")
print("=" * 60)

date_validation = pd.DataFrame({
    "Column": [
        "appointment_datetime",
        "booked_datetime"
    ],
    "Invalid_or_Missing": [
        appointment_dates.isna().sum(),
        booking_dates.isna().sum()
    ],
    "Valid_Dates": [
        appointment_dates.notna().sum(),
        booking_dates.notna().sum()
    ]
})

display(date_validation)


# 3. EXAMINE DATE RANGES

print("\nDATE RANGE CHECK")
print("=" * 60)

date_ranges = pd.DataFrame({
    "Column": [
        "appointment_datetime",
        "booked_datetime"
    ],
    "Earliest_Date": [
        appointment_dates.min(),
        booking_dates.min()
    ],
    "Latest_Date": [
        appointment_dates.max(),
        booking_dates.max()
    ]
})

display(date_ranges)


# 4. CALCULATE BOOKING LEAD TIME

# Lead time = appointment date - booking date
# Positive values indicate booking before appointment.

booking_lead_days = (
    appointment_dates - booking_dates
).dt.total_seconds() / 86400


# 5. CHECK INVALID BOOKING SEQUENCES

print("\nBOOKING SEQUENCE CHECK")
print("=" * 60)

negative_lead = booking_lead_days < 0
zero_lead = booking_lead_days == 0
positive_lead = booking_lead_days > 0

print(
    "Appointments booked after scheduled time:",
    negative_lead.sum()
)

print(
    "Appointments booked at exactly scheduled time:",
    zero_lead.sum()
)

print(
    "Appointments booked before scheduled time:",
    positive_lead.sum()
)

print(
    "Appointments with unavailable lead time:",
    booking_lead_days.isna().sum()
)


# 6. BOOKING LEAD TIME STATISTICAL SUMMARY

print("\nBOOKING LEAD TIME STATISTICAL SUMMARY")
print("=" * 60)

lead_time_summary = booking_lead_days.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

display(
    lead_time_summary.to_frame(
        name="Booking_Lead_Days"
    )
)


# 7. INVESTIGATE UNUSUAL BOOKING INTERVALS

print("\nUNUSUAL BOOKING INTERVAL CHECK")
print("=" * 60)

print(
    "Negative lead times:",
    (booking_lead_days < 0).sum()
)

print(
    "Lead times greater than 180 days:",
    (booking_lead_days > 180).sum()
)

print(
    "Lead times greater than 365 days:",
    (booking_lead_days > 365).sum()
)



# 8. DISPLAY SAMPLE PROBLEMATIC RECORDS

print("\nSAMPLE RECORDS WITH NEGATIVE LEAD TIMES")
print("=" * 60)

problematic_dates = df.loc[
    negative_lead,
    [
        "appointment_id",
        "patient_id",
        "appointment_datetime",
        "booked_datetime",
        "attendance_status"
    ]
].copy()

display(problematic_dates.head(20))


# 9. INVESTIGATE APPOINTMENT DAYS

print("\nAPPOINTMENT DAY DISTRIBUTION")
print("=" * 60)

appointment_days = (
    appointment_dates.dt.day_name()
    .value_counts(dropna=False)
)

display(
    appointment_days.to_frame(
        name="Appointment_Count"
    )
)


# 10. INVESTIGATE APPOINTMENT HOURS

print("\nAPPOINTMENT HOUR DISTRIBUTION")
print("=" * 60)

appointment_hours = (
    appointment_dates.dt.hour
    .value_counts(dropna=False)
    .sort_index()
)

display(
    appointment_hours.to_frame(
        name="Appointment_Count"
    )
)


# 11. SAVE PROFILING RESULTS

date_validation.to_csv(
    REPORT_DIR / "date_validation_summary.csv",
    index=False
)

date_ranges.to_csv(
    REPORT_DIR / "date_range_summary.csv",
    index=False
)

lead_time_summary.to_csv(
    REPORT_DIR / "booking_lead_time_summary.csv"
)

problematic_dates.to_csv(
    REPORT_DIR / "negative_booking_lead_records.csv",
    index=False
)

appointment_days.to_csv(
    REPORT_DIR / "appointment_day_distribution.csv"
)

appointment_hours.to_csv(
    REPORT_DIR / "appointment_hour_distribution.csv"
)


# 12. CONFIRM COMPLETION

print("\nCELL 8 COMPLETED SUCCESSFULLY")
print("Date and booking lead-time profiling reports saved.")
print("Original dataset remains unchanged.")

APPOINTMENT DATE AND BOOKING LEAD TIME PROFILING

DATE VALIDATION CHECK


,Column,Invalid_or_Missing,Valid_Dates
0,appointment_datetime,0,40180
1,booked_datetime,0,40180



DATE RANGE CHECK


,Column,Earliest_Date,Latest_Date
0,appointment_datetime,2025-04-01 08:30:00,2026-03-31 16:30:00
1,booked_datetime,2024-06-21 17:47:00,2026-04-13 09:00:00



BOOKING SEQUENCE CHECK
Appointments booked after scheduled time: 110
Appointments booked at exactly scheduled time: 0
Appointments booked before scheduled time: 40070
Appointments with unavailable lead time: 0

BOOKING LEAD TIME STATISTICAL SUMMARY


,Booking_Lead_Days
count,40180.000000
mean,45.797164
std,37.535955
min,-19.000000
1%,1.193861
5%,4.976354
25%,18.224132
50%,36.073958
75%,62.917882
95%,119.168368


... (3 outputs left out)

,Appointment_Count
appointment_datetime,
Monday,17220
Friday,5794
Wednesday,5754
Tuesday,5711
Thursday,5701



APPOINTMENT HOUR DISTRIBUTION


,Appointment_Count
appointment_datetime,
8,2286
9,4735
10,4750
11,4874
12,4755
13,4694
14,4775
15,4670
16,4641



CELL 8 COMPLETED SUCCESSFULLY
Date and booking lead-time profiling reports saved.
Original dataset remains unchanged.


In [12]:

# DNA RATE ANALYSIS BY APPOINTMENT DAY, HOUR
#         AND BOOKING LEAD TIME


# Purpose:
# 1. Calculate overall DNA rate
# 2. Investigate DNA rates by appointment weekday
# 3. Investigate DNA rates by appointment hour
# 4. Investigate DNA rates by booking lead time
# 5. Identify potential predictive patterns



import pandas as pd
import numpy as np

print("DNA RATE ANALYSIS")
print("=" * 60)


# 1. CREATE TEMPORARY ANALYSIS COPY

# Remove exact duplicate records from the temporary copy.
# The original df remains unchanged.

df_analysis = df.drop_duplicates().copy()

print("\nTEMPORARY DATASET CHECK")
print("=" * 60)

print("Original records:", len(df))
print("Temporary analysis records:", len(df_analysis))
print(
    "Duplicate records excluded:",
    len(df) - len(df_analysis)
)

print(
    "Repeated appointment IDs remaining:",
    df_analysis["appointment_id"].duplicated().sum()
)


# 2. PREPARE ATTENDANCE DATA

# Only Attended and DNA are included in this analysis.
# Cancelled appointments are excluded from the denominator.

df_dna = df_analysis[
    df_analysis["attendance_status"].isin(
        ["Attended", "DNA"]
    )
].copy()

# Temporary binary outcome for exploratory analysis

df_dna["dna_flag"] = (
    df_dna["attendance_status"] == "DNA"
).astype(int)


# 3. OVERALL DNA RATE

print("\nOVERALL DNA RATE")
print("=" * 60)

total_appointments = len(df_dna)
total_dnas = df_dna["dna_flag"].sum()

overall_dna_rate = (
    total_dnas / total_appointments * 100
)

print("Eligible appointments:", total_appointments)
print("Total DNAs:", total_dnas)
print("Overall DNA rate (%):", round(overall_dna_rate, 2))


# 4. CONVERT DATES IN TEMPORARY COPY

df_dna["appointment_date"] = pd.to_datetime(
    df_dna["appointment_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

df_dna["booking_date"] = pd.to_datetime(
    df_dna["booked_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)


# 5. EXTRACT APPOINTMENT DAY AND HOUR

df_dna["appointment_day"] = (
    df_dna["appointment_date"].dt.day_name()
)

df_dna["appointment_hour"] = (
    df_dna["appointment_date"].dt.hour
)


# 6. DNA RATE BY APPOINTMENT DAY

print("\nDNA RATE BY APPOINTMENT DAY")
print("=" * 60)

dna_by_day = (
    df_dna.groupby("appointment_day")
    .agg(
        Total_Appointments=("dna_flag", "size"),
        DNA_Count=("dna_flag", "sum")
    )
)

dna_by_day["DNA_Rate_Percentage"] = (
    dna_by_day["DNA_Count"]
    / dna_by_day["Total_Appointments"]
    * 100
).round(2)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

dna_by_day = dna_by_day.reindex(day_order)

display(dna_by_day)


# 7. DNA RATE BY APPOINTMENT HOUR

print("\nDNA RATE BY APPOINTMENT HOUR")
print("=" * 60)

dna_by_hour = (
    df_dna.groupby("appointment_hour")
    .agg(
        Total_Appointments=("dna_flag", "size"),
        DNA_Count=("dna_flag", "sum")
    )
)

dna_by_hour["DNA_Rate_Percentage"] = (
    dna_by_hour["DNA_Count"]
    / dna_by_hour["Total_Appointments"]
    * 100
).round(2)

display(dna_by_hour)


# 8. CALCULATE BOOKING LEAD TIME

df_dna["booking_lead_days"] = (
    df_dna["appointment_date"]
    - df_dna["booking_date"]
).dt.total_seconds() / 86400


# 9. CREATE TEMPORARY BOOKING LEAD TIME GROUPS

# Negative lead times are kept as a separate category.
# No records are deleted.

lead_bins = [
    -np.inf,
    0,
    7,
    14,
    30,
    60,
    90,
    180,
    np.inf
]

lead_labels = [
    "Negative lead time",
    "0-7 days",
    "7-14 days",
    "14-30 days",
    "30-60 days",
    "60-90 days",
    "90-180 days",
    "Over 180 days"
]

df_dna["lead_time_group"] = pd.cut(
    df_dna["booking_lead_days"],
    bins=lead_bins,
    labels=lead_labels,
    right=True,
    include_lowest=True
)


# 10. DNA RATE BY BOOKING LEAD TIME

print("\nDNA RATE BY BOOKING LEAD TIME")
print("=" * 60)

dna_by_lead_time = (
    df_dna.groupby(
        "lead_time_group",
        observed=True
    )
    .agg(
        Total_Appointments=("dna_flag", "size"),
        DNA_Count=("dna_flag", "sum")
    )
)

dna_by_lead_time["DNA_Rate_Percentage"] = (
    dna_by_lead_time["DNA_Count"]
    / dna_by_lead_time["Total_Appointments"]
    * 100
).round(2)

display(dna_by_lead_time)


# 11. SAVE PROFILING RESULTS

dna_by_day.to_csv(
    REPORT_DIR / "dna_rate_by_appointment_day.csv"
)

dna_by_hour.to_csv(
    REPORT_DIR / "dna_rate_by_appointment_hour.csv"
)

dna_by_lead_time.to_csv(
    REPORT_DIR / "dna_rate_by_booking_lead_time.csv"
)


# 12. CONFIRM COMPLETION

print("\nCELL 9 COMPLETED SUCCESSFULLY")
print("DNA rate analysis reports saved.")
print("Original dataset remains unchanged.")

DNA RATE ANALYSIS

TEMPORARY DATASET CHECK
Original records: 40180
Temporary analysis records: 40000
Duplicate records excluded: 180
Repeated appointment IDs remaining: 0

OVERALL DNA RATE
Eligible appointments: 36364
Total DNAs: 3316
Overall DNA rate (%): 9.12

DNA RATE BY APPOINTMENT DAY


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
appointment_day,,,
Monday,15603,1522,9.75
Tuesday,5149,430,8.35
Wednesday,5221,423,8.10
Thursday,5163,466,9.03
Friday,5228,475,9.09



DNA RATE BY APPOINTMENT HOUR


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
appointment_hour,,,
8,2086,220,10.55
9,4278,434,10.14
10,4287,395,9.21
11,4410,392,8.89
12,4335,366,8.44
13,4232,385,9.10
14,4323,383,8.86
15,4213,368,8.73
16,4200,373,8.88



DNA RATE BY BOOKING LEAD TIME


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
lead_time_group,,,
Negative lead time,100,10,10.00
0-7 days,2697,195,7.23
7-14 days,3788,310,8.18
14-30 days,8643,657,7.60
30-60 days,11308,1020,9.02
60-90 days,5568,556,9.99
90-180 days,3950,508,12.86
Over 180 days,310,60,19.35



CELL 9 COMPLETED SUCCESSFULLY
DNA rate analysis reports saved.
Original dataset remains unchanged.


In [13]:

# DNA RATE ANALYSIS BY PATIENT CHARACTERISTICS
#          AND CLINICAL FACTORS


# PURPOSE:
# 1. Examine DNA rates across patient characteristics
# 2. Examine DNA rates across clinical and appointment factors
# 3. Identify potential predictors of missed appointments
# 4. Support future fairness and bias assessment
# 5. Save profiling results



import pandas as pd
import numpy as np

print("DNA RATE ANALYSIS BY PATIENT AND CLINICAL FACTORS")
print("=" * 65)



# 1. CREATE TEMPORARY ANALYSIS DATASET


df_profile = df.drop_duplicates().copy()

# Verify that each appointment is represented once.

assert df_profile["appointment_id"].is_unique, (
    "Repeated appointment IDs remain. Investigate before continuing."
)

# Exclude cancellations from this particular DNA analysis.

df_profile = df_profile[
    df_profile["attendance_status"].isin(
        ["Attended", "DNA"]
    )
].copy()

# Temporary binary outcome.

df_profile["dna_flag"] = (
    df_profile["attendance_status"] == "DNA"
).astype(int)

print("\nANALYSIS POPULATION")
print("-" * 65)

print("Eligible appointments:", len(df_profile))
print("Total DNAs:", df_profile["dna_flag"].sum())

print(
    "Overall DNA rate:",
    round(df_profile["dna_flag"].mean() * 100, 2),
    "%"
)



# 2. FUNCTION TO CALCULATE DNA RATES

def calculate_dna_rates(data, column):

    result = (
        data.groupby(
            column,
            dropna=False,
            observed=True
        )
        .agg(
            Total_Appointments=("dna_flag", "size"),
            DNA_Count=("dna_flag", "sum")
        )
    )

    result["DNA_Rate_Percentage"] = (
        result["DNA_Count"]
        / result["Total_Appointments"]
        * 100
    ).round(2)

    return result.sort_values(
        "DNA_Rate_Percentage",
        ascending=False
    )



# 3. AGE GROUP ANALYSIS


print("\nDNA RATE BY AGE GROUP")
print("=" * 65)

# Invalid ages remain visible as a separate group.
# They are not deleted or corrected.

age_bins = [
    -np.inf,
    -0.001,
    17,
    34,
    49,
    64,
    79,
    110,
    np.inf
]

age_labels = [
    "Invalid: negative age",
    "0-17",
    "18-34",
    "35-49",
    "50-64",
    "65-79",
    "80-110",
    "Over 110"
]

df_profile["age_group"] = pd.cut(
    df_profile["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

dna_by_age = calculate_dna_rates(
    df_profile,
    "age_group"
)

display(dna_by_age)



# 4. SEX


print("\nDNA RATE BY SEX")
print("=" * 65)

dna_by_sex = calculate_dna_rates(
    df_profile,
    "sex"
)

display(dna_by_sex)



# 5. ETHNIC GROUP


print("\nDNA RATE BY ETHNIC GROUP")
print("=" * 65)

dna_by_ethnicity = calculate_dna_rates(
    df_profile,
    "ethnic_group"
)

display(dna_by_ethnicity)



# 6. DEPRIVATION (IMD DECILE)


print("\nDNA RATE BY IMD DECILE")
print("=" * 65)

# Missing deprivation values remain a separate category.

dna_by_imd = calculate_dna_rates(
    df_profile,
    "imd_decile"
)

display(dna_by_imd)



# 7. PREVIOUS DNAs


print("\nDNA RATE BY PREVIOUS DNAs IN LAST 12 MONTHS")
print("=" * 65)

dna_by_previous_dnas = calculate_dna_rates(
    df_profile,
    "previous_dnas_12m"
)

display(dna_by_previous_dnas)



# 8. LONG-TERM CONDITIONS


print("\nDNA RATE BY LONG-TERM CONDITIONS")
print("=" * 65)

dna_by_conditions = calculate_dna_rates(
    df_profile,
    "long_term_conditions"
)

display(dna_by_conditions)



# 9. INTERPRETER REQUIREMENT


print("\nDNA RATE BY INTERPRETER REQUIREMENT")
print("=" * 65)

dna_by_interpreter = calculate_dna_rates(
    df_profile,
    "interpreter_required"
)

display(dna_by_interpreter)



# 10. SPECIALTY


print("\nDNA RATE BY SPECIALTY")
print("=" * 65)

dna_by_specialty = calculate_dna_rates(
    df_profile,
    "specialty"
)

display(dna_by_specialty)



# 11. APPOINTMENT TYPE


print("\nDNA RATE BY APPOINTMENT TYPE")
print("=" * 65)

dna_by_appointment_type = calculate_dna_rates(
    df_profile,
    "appointment_type"
)

display(dna_by_appointment_type)



# 12. CONSULTATION MODE


print("\nDNA RATE BY CONSULTATION MODE")
print("=" * 65)

dna_by_consultation = calculate_dna_rates(
    df_profile,
    "consultation_mode"
)

display(dna_by_consultation)



# 13. REFERRAL SOURCE


print("\nDNA RATE BY REFERRAL SOURCE")
print("=" * 65)

dna_by_referral = calculate_dna_rates(
    df_profile,
    "referral_source"
)

display(dna_by_referral)


# 14. SMS REMINDER


print("\nDNA RATE BY SMS REMINDER STATUS")
print("=" * 65)

dna_by_sms = calculate_dna_rates(
    df_profile,
    "sms_reminder_sent"
)

display(dna_by_sms)



# 15. RESCHEDULED COUNT


print("\nDNA RATE BY RESCHEDULED COUNT")
print("=" * 65)

dna_by_rescheduled = calculate_dna_rates(
    df_profile,
    "rescheduled_count"
)

display(dna_by_rescheduled)



# 16. SAVE PROFILING RESULTS


profiling_results = {
    "dna_by_age": dna_by_age,
    "dna_by_sex": dna_by_sex,
    "dna_by_ethnicity": dna_by_ethnicity,
    "dna_by_imd": dna_by_imd,
    "dna_by_previous_dnas": dna_by_previous_dnas,
    "dna_by_conditions": dna_by_conditions,
    "dna_by_interpreter": dna_by_interpreter,
    "dna_by_specialty": dna_by_specialty,
    "dna_by_appointment_type": dna_by_appointment_type,
    "dna_by_consultation": dna_by_consultation,
    "dna_by_referral": dna_by_referral,
    "dna_by_sms": dna_by_sms,
    "dna_by_rescheduled": dna_by_rescheduled
}

for filename, result in profiling_results.items():

    result.to_csv(
        REPORT_DIR / f"{filename}.csv"
    )



# 17. COMPLETION


print("\nCELL 10 COMPLETED SUCCESSFULLY")
print("Patient and clinical DNA profiling reports saved.")
print("Original dataset remains unchanged.")

DNA RATE ANALYSIS BY PATIENT AND CLINICAL FACTORS

ANALYSIS POPULATION
-----------------------------------------------------------------
Eligible appointments: 36364
Total DNAs: 3316
Overall DNA rate: 9.12 %

DNA RATE BY AGE GROUP


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
age_group,,,
18-34,5869,707,12.05
0-17,4430,462,10.43
80-110,5032,504,10.02
35-49,5062,504,9.96
50-64,5280,450,8.52
Over 110,25,2,8.00
65-79,10636,686,6.45
Invalid: negative age,30,1,3.33



DNA RATE BY SEX


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
sex,,,
F,333,38,11.41
M,302,33,10.93
Unknown,88,9,10.23
male,258,25,9.69
Female,18579,1727,9.30
female,286,26,9.09
Male,16518,1458,8.83



DNA RATE BY ETHNIC GROUP


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
ethnic_group,,,
Black or Black British,1936,223,11.52
Other ethnic group,883,94,10.65
Asian or Asian British,4490,428,9.53
Not Stated,1308,122,9.33
White,26557,2350,8.85
Mixed,1190,99,8.32


... (17 outputs left out)

,Total_Appointments,DNA_Count,DNA_Rate_Percentage
sms_reminder_sent,,,
N,8709,986,11.32
Y,27655,2330,8.43



DNA RATE BY RESCHEDULED COUNT


,Total_Appointments,DNA_Count,DNA_Rate_Percentage
rescheduled_count,,,
3,693,91,13.13
2,2198,248,11.28
1,6672,671,10.06
0,26801,2306,8.60



CELL 10 COMPLETED SUCCESSFULLY
Patient and clinical DNA profiling reports saved.
Original dataset remains unchanged.


In [14]:

# TRAVEL DISTANCE AND MISSING VALUES ANALYSIS



import pandas as pd
import numpy as np

print("=" * 65)
print("CELL 11: TRAVEL DISTANCE AND MISSING VALUES ANALYSIS")
print("=" * 65)



# 1. CREATE TEMPORARY ANALYSIS DATASET


# Remove exact duplicates temporarily for analysis.
# Keep only Attended and DNA outcomes.
# The original dataframe (df) remains unchanged.

df_eda11 = df.drop_duplicates().copy()

df_eda11 = df_eda11[
    df_eda11["attendance_status"].isin(["Attended", "DNA"])
].copy()

df_eda11["dna_flag"] = (
    df_eda11["attendance_status"] == "DNA"
).astype(int)

print("\n1. ANALYSIS POPULATION")
print("-" * 65)

print(f"Original dataset rows: {len(df):,}")
print(f"Analysis appointments: {len(df_eda11):,}")
print(f"Total DNAs: {df_eda11['dna_flag'].sum():,}")

print(
    f"Overall DNA rate: "
    f"{df_eda11['dna_flag'].mean() * 100:.2f}%"
)



# 2. TRAVEL DISTANCE DISTRIBUTION


print("\n2. TRAVEL DISTANCE DISTRIBUTION")
print("-" * 65)

display(
    df_eda11["distance_km"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

distance_checks = pd.DataFrame({
    "Check": [
        "Missing distances",
        "Negative distances",
        "Zero distances",
        "Distances above 50 km",
        "Distances above 100 km",
        "Distances above 200 km"
    ],
    "Count": [
        df_eda11["distance_km"].isna().sum(),
        (df_eda11["distance_km"] < 0).sum(),
        (df_eda11["distance_km"] == 0).sum(),
        (df_eda11["distance_km"] > 50).sum(),
        (df_eda11["distance_km"] > 100).sum(),
        (df_eda11["distance_km"] > 200).sum()
    ]
})

display(distance_checks)



# 3. DNA RATE BY TRAVEL DISTANCE


print("\n3. DNA RATE BY TRAVEL DISTANCE")
print("-" * 65)

# Temporary distance categories for exploratory analysis.

distance_bins = [
    -np.inf, 0, 5, 10, 20, 50, 100, np.inf
]

distance_labels = [
    "Negative distance",
    "0-5 km",
    ">5-10 km",
    ">10-20 km",
    ">20-50 km",
    ">50-100 km",
    ">100 km"
]

df_eda11["distance_group"] = pd.cut(
    df_eda11["distance_km"],
    bins=distance_bins,
    labels=distance_labels,
    include_lowest=False
)

# Ensure zero distances are included in the 0-5 km group.
df_eda11.loc[
    df_eda11["distance_km"] == 0,
    "distance_group"
] = "0-5 km"

df_eda11["distance_group"] = (
    df_eda11["distance_group"]
    .cat.add_categories(["Missing"])
    .fillna("Missing")
)

distance_summary = (
    df_eda11.groupby("distance_group", observed=True)
    .agg(
        Appointments=("dna_flag", "size"),
        DNAs=("dna_flag", "sum")
    )
)

distance_summary["DNA_Rate_Percent"] = (
    distance_summary["DNAs"]
    / distance_summary["Appointments"]
    * 100
).round(2)

display(distance_summary)


# 4. MISSING VALUES IN THE ORIGINAL DATASET


print("\n4. MISSING VALUES IN ORIGINAL DATASET")
print("-" * 65)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (
        df.isna().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary[
        missing_summary["Missing_Count"] > 0
    ]
    .sort_values("Missing_Count", ascending=False)
)

display(missing_summary)



# 5. MISSING TRAVEL DISTANCE VS DNA


print("\n5. MISSING TRAVEL DISTANCE VS DNA")
print("-" * 65)

df_eda11["distance_missing"] = np.where(
    df_eda11["distance_km"].isna(),
    "Missing",
    "Available"
)

distance_missing_summary = (
    df_eda11.groupby("distance_missing")
    .agg(
        Appointments=("dna_flag", "size"),
        DNAs=("dna_flag", "sum")
    )
)

distance_missing_summary["DNA_Rate_Percent"] = (
    distance_missing_summary["DNAs"]
    / distance_missing_summary["Appointments"]
    * 100
).round(2)

display(distance_missing_summary)



# 6. MISSING DEPRIVATION INFORMATION VS DNA


print("\n6. MISSING IMD DECILE VS DNA")
print("-" * 65)

df_eda11["imd_missing"] = np.where(
    df_eda11["imd_decile"].isna(),
    "Missing",
    "Available"
)

imd_missing_summary = (
    df_eda11.groupby("imd_missing")
    .agg(
        Appointments=("dna_flag", "size"),
        DNAs=("dna_flag", "sum")
    )
)

imd_missing_summary["DNA_Rate_Percent"] = (
    imd_missing_summary["DNAs"]
    / imd_missing_summary["Appointments"]
    * 100
).round(2)

display(imd_missing_summary)



# 7. OVERLAPPING MISSING VALUES


print("\n7. OVERLAPPING MISSING VALUES")
print("-" * 65)

distance_missing = df_eda11["distance_km"].isna()
imd_missing = df_eda11["imd_decile"].isna()

print(
    "Appointments missing both distance and IMD:",
    (distance_missing & imd_missing).sum()
)

print(
    "Appointments missing either distance or IMD:",
    (distance_missing | imd_missing).sum()
)



# 8. SAVE ANALYSIS REPORTS


distance_summary.to_csv(
    REPORT_DIR / "cell11_distance_dna_summary.csv"
)

missing_summary.to_csv(
    REPORT_DIR / "cell11_missing_values_summary.csv"
)

distance_missing_summary.to_csv(
    REPORT_DIR / "cell11_distance_missing_dna.csv"
)

imd_missing_summary.to_csv(
    REPORT_DIR / "cell11_imd_missing_dna.csv"
)

print("\nAnalysis reports saved successfully.")



# 9. FINAL DATA INTEGRITY CHECK


print("\n9. DATA INTEGRITY CHECK")
print("-" * 65)

print(f"Original dataset rows: {len(df):,}")
print(f"Original dataset columns: {df.shape[1]}")

assert len(df) == 40180
assert df.shape[1] == 20

print("\nCELL 11 COMPLETED SUCCESSFULLY.")
print("Original dataset remains unchanged.")
print("No permanent cleaning or feature engineering performed.")

CELL 11: TRAVEL DISTANCE AND MISSING VALUES ANALYSIS

1. ANALYSIS POPULATION
-----------------------------------------------------------------
Original dataset rows: 40,180
Analysis appointments: 36,364
Total DNAs: 3,316
Overall DNA rate: 9.12%

2. TRAVEL DISTANCE DISTRIBUTION
-----------------------------------------------------------------


count    36001.000000
mean         8.342885
std          5.674968
min          0.300000
25%          4.200000
50%          7.000000
75%         11.100000
90%         15.800000
95%         19.300000
99%         27.600000
max         47.400000
Name: distance_km, dtype: float64

,Check,Count
0,Missing distances,363
1,Negative distances,0
2,Zero distances,0
3,Distances above 50 km,0
4,Distances above 100 km,0
5,Distances above 200 km,0



3. DNA RATE BY TRAVEL DISTANCE
-----------------------------------------------------------------


,Appointments,DNAs,DNA_Rate_Percent
distance_group,,,
0-5 km,11956,998,8.35
>5-10 km,13288,1214,9.14
>10-20 km,9194,913,9.93
>20-50 km,1563,163,10.43
Missing,363,28,7.71



4. MISSING VALUES IN ORIGINAL DATASET
-----------------------------------------------------------------


... (2 outputs left out)

,Appointments,DNAs,DNA_Rate_Percent
distance_missing,,,
Available,36001,3288,9.13
Missing,363,28,7.71



6. MISSING IMD DECILE VS DNA
-----------------------------------------------------------------


,Appointments,DNAs,DNA_Rate_Percent
imd_missing,,,
Available,35096,3210,9.15
Missing,1268,106,8.36



7. OVERLAPPING MISSING VALUES
-----------------------------------------------------------------
Appointments missing both distance and IMD: 15
Appointments missing either distance or IMD: 1616

Analysis reports saved successfully.

9. DATA INTEGRITY CHECK
-----------------------------------------------------------------
Original dataset rows: 40,180
Original dataset columns: 20

CELL 11 COMPLETED SUCCESSFULLY.
Original dataset remains unchanged.
No permanent cleaning or feature engineering performed.


In [15]:
# DATA CLEANING PREPARATION AND DECISION LOG


import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 12: DATA CLEANING PREPARATION AND DECISION LOG")
print("=" * 70)


# 1. PRESERVE ORIGINAL DATA

df_cleaning_review = df.copy(deep=True)

original_rows, original_columns = df.shape

print("\n1. ORIGINAL DATASET")
print("-" * 70)

print(f"Original rows: {original_rows:,}")
print(f"Original columns: {original_columns}")


# 2. VERIFY EXACT DUPLICATES

print("\n2. EXACT DUPLICATE CHECK")
print("-" * 70)

duplicate_count = df_cleaning_review.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")

# Temporary deduplicated master dataset.
df_master_review = df_cleaning_review.drop_duplicates().copy()

print(f"Unique appointment records: {len(df_master_review):,}")

print(
    "Duplicate appointment IDs remaining:",
    df_master_review["appointment_id"].duplicated().sum()
)


# 3. VERIFY ATTENDANCE OUTCOMES

print("\n3. ATTENDANCE STATUS AFTER DEDUPLICATION")
print("-" * 70)

attendance_summary = (
    df_master_review["attendance_status"]
    .value_counts(dropna=False)
    .rename_axis("Attendance Status")
    .reset_index(name="Appointments")
)

attendance_summary["Percentage"] = (
    attendance_summary["Appointments"]
    / len(df_master_review) * 100
).round(2)

display(attendance_summary)


# 4. SEPARATE MODEL POPULATION

print("\n4. MODEL POPULATION")
print("-" * 70)

# Keeping cancellation records in the master review dataset.
# Only attended and DNA appointments enter the binary model review.

df_model_review = df_master_review[
    df_master_review["attendance_status"].isin(
        ["Attended", "DNA"]
    )
].copy()

df_cancelled_review = df_master_review[
    df_master_review["attendance_status"].isin(
        ["Cancelled by patient", "Cancelled by hospital"]
    )
].copy()

print(f"Master records: {len(df_master_review):,}")
print(f"Model-eligible records: {len(df_model_review):,}")
print(f"Cancelled records: {len(df_cancelled_review):,}")

print("\nCancellation breakdown:")

display(
    df_cancelled_review["attendance_status"]
    .value_counts()
    .rename_axis("Cancellation Type")
    .reset_index(name="Appointments")
)

assert (
    len(df_model_review) + len(df_cancelled_review)
    == len(df_master_review)
), "Some attendance outcomes are not accounted for."


# 5. IDENTIFY RECORDS REQUIRING CLEANING

print("\n5. DATA QUALITY ISSUES")
print("-" * 70)

appointment_dates = pd.to_datetime(
    df_master_review["appointment_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

booking_dates = pd.to_datetime(
    df_master_review["booked_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

lead_days = (
    appointment_dates - booking_dates
).dt.total_seconds() / 86400

quality_checks = pd.DataFrame({
    "Issue": [
        "Exact duplicate rows in original data",
        "Negative ages",
        "Ages above 110",
        "Missing IMD decile",
        "Missing travel distance",
        "Negative travel distance",
        "Invalid appointment dates",
        "Invalid booking dates",
        "Negative booking lead time",
        "Booking lead time above 180 days",
        "Missing clinic check-in time"
    ],
    "Count": [
        duplicate_count,
        (df_master_review["age"] < 0).sum(),
        (df_master_review["age"] > 110).sum(),
        df_master_review["imd_decile"].isna().sum(),
        df_master_review["distance_km"].isna().sum(),
        (df_master_review["distance_km"] < 0).sum(),
        appointment_dates.isna().sum(),
        booking_dates.isna().sum(),
        (lead_days < 0).sum(),
        (lead_days > 180).sum(),
        df_master_review["clinic_checkin_time"].isna().sum()
    ]
})

display(quality_checks)


# 6. DOCUMENT PROPOSED CLEANING DECISIONS

print("\n6. PROPOSED CLEANING DECISIONS")
print("-" * 70)

cleaning_decisions = pd.DataFrame({
    "Data Issue": [
        "Exact duplicate records",
        "Cancelled appointments",
        "Invalid ages",
        "Inconsistent sex categories",
        "Specialty spelling inconsistencies",
        "Missing IMD decile",
        "Missing travel distance",
        "Negative booking lead time",
        "Very long booking lead time",
        "Clinic check-in time"
    ],

    "Proposed Action": [
        "Remove exact duplicates from cleaned master dataset",
        "Retain in master dataset; exclude from binary model training",
        "Investigate; treat confirmed invalid values as missing",
        "Standardise equivalent category labels",
        "Correct confirmed spelling inconsistencies",
        "Retain records; assess missingness and suitable imputation",
        "Retain records; assess missingness and suitable imputation",
        "Investigate chronology before deciding whether to exclude",
        "Retain initially; investigate extreme values",
        "Exclude from model predictors"
    ],

    "Reason": [
        "Avoid counting the same appointment more than once",
        "Cancellations are different outcomes from DNA",
        "Invalid ages may distort analysis and predictions",
        "Equivalent labels should represent the same category",
        "Spelling errors create artificial categories",
        "Avoid unnecessary loss of patient records",
        "Avoid unnecessary loss of patient records",
        "Booking after appointment may indicate a data error",
        "Long waiting times may be valid and predictive",
        "Check-in information is unavailable before attendance"
    ],

    "Status": ["Proposed - not applied"] * 10
})

display(cleaning_decisions)


# 7. SAVE REVIEW REPORTS

print("\n7. SAVE REPORTS")
print("-" * 70)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

attendance_summary.to_csv(
    REPORT_DIR / "cell12_attendance_summary.csv",
    index=False
)

quality_checks.to_csv(
    REPORT_DIR / "cell12_quality_checks.csv",
    index=False
)

cleaning_decisions.to_csv(
    REPORT_DIR / "cell12_cleaning_decision_log.csv",
    index=False
)

print("Cell 12 reports saved successfully.")


# 8. VERIFY ORIGINAL DATA REMAINS UNCHANGED

print("\n8. FINAL DATA INTEGRITY CHECK")
print("-" * 70)

assert df.shape == (original_rows, original_columns)
assert df.equals(df_cleaning_review)

print(f"Original dataset: {df.shape[0]:,} rows")
print(f"Original columns: {df.shape[1]}")
print(f"Temporary master review: {len(df_master_review):,} rows")
print(f"Temporary model review: {len(df_model_review):,} rows")

print("\nCELL 12 COMPLETED SUCCESSFULLY.")
print("Original dataset remains unchanged.")
print("Cleaning decisions documented but not yet applied.")

CELL 12: DATA CLEANING PREPARATION AND DECISION LOG

1. ORIGINAL DATASET
----------------------------------------------------------------------
Original rows: 40,180
Original columns: 20

2. EXACT DUPLICATE CHECK
----------------------------------------------------------------------
Exact duplicate rows: 180
Unique appointment records: 40,000
Duplicate appointment IDs remaining: 0

3. ATTENDANCE STATUS AFTER DEDUPLICATION
----------------------------------------------------------------------


,Attendance Status,Appointments,Percentage
0,Attended,33048,82.62
1,DNA,3316,8.29
2,Cancelled by patient,2421,6.05
3,Cancelled by hospital,1215,3.04



4. MODEL POPULATION
----------------------------------------------------------------------
Master records: 40,000
Model-eligible records: 36,364
Cancelled records: 3,636

Cancellation breakdown:


,Cancellation Type,Appointments
0,Cancelled by patient,2421
1,Cancelled by hospital,1215



5. DATA QUALITY ISSUES
----------------------------------------------------------------------


,Issue,Count
0,Exact duplicate rows in original data,180
1,Negative ages,35
2,Ages above 110,25
3,Missing IMD decile,1400
4,Missing travel distance,400
5,Negative travel distance,0
6,Invalid appointment dates,0
7,Invalid booking dates,0
8,Negative booking lead time,110
9,Booking lead time above 180 days,338



6. PROPOSED CLEANING DECISIONS
----------------------------------------------------------------------


,Data Issue,Proposed Action,Reason,Status
0,Exact duplicate records,Remove exact duplicates from cleaned master da...,Avoid counting the same appointment more than ...,Proposed - not applied
1,Cancelled appointments,Retain in master dataset; exclude from binary ...,Cancellations are different outcomes from DNA,Proposed - not applied
2,Invalid ages,Investigate; treat confirmed invalid values as...,Invalid ages may distort analysis and predictions,Proposed - not applied
3,Inconsistent sex categories,Standardise equivalent category labels,Equivalent labels should represent the same ca...,Proposed - not applied
4,Specialty spelling inconsistencies,Correct confirmed spelling inconsistencies,Spelling errors create artificial categories,Proposed - not applied
5,Missing IMD decile,Retain records; assess missingness and suitabl...,Avoid unnecessary loss of patient records,Proposed - not applied
6,Missing travel distance,Retain records; assess missingness and suitabl...,Avoid unnecessary loss of patient records,Proposed - not applied
7,Negative booking lead time,Investigate chronology before deciding whether...,Booking after appointment may indicate a data ...,Proposed - not applied
8,Very long booking lead time,Retain initially; investigate extreme values,Long waiting times may be valid and predictive,Proposed - not applied
9,Clinic check-in time,Exclude from model predictors,Check-in information is unavailable before att...,Proposed - not applied



7. SAVE REPORTS
----------------------------------------------------------------------
Cell 12 reports saved successfully.

8. FINAL DATA INTEGRITY CHECK
----------------------------------------------------------------------
Original dataset: 40,180 rows
Original columns: 20
Temporary master review: 40,000 rows
Temporary model review: 36,364 rows

CELL 12 COMPLETED SUCCESSFULLY.
Original dataset remains unchanged.
Cleaning decisions documented but not yet applied.
